# Week 8 Workshop – Retail Sales & Customer Data Analysis





---
# Part 1: Data Loading and Exploration

## 1.1 – Loading Data

In [ ]:
import pandas as pd
import numpy as np

# Load all five datasets
customers = pd.read_csv('wk8-customers.csv')
products  = pd.read_csv('wk8-products.csv')
stores    = pd.read_csv('wk8-stores.csv')
sales     = pd.read_csv('wk8-sales.csv')
inventory = pd.read_csv('wk8-inventory.csv')

print("Datasets loaded successfully\n")
for df, name in [(customers,'customers'), (products,'products'), (stores,'stores'),
                 (sales,'sales'), (inventory,'inventory')]:
    print(f"  {name:<12}  {df.shape[0]:>3} rows × {df.shape[1]} columns")


## 1.2 – Exploratory Analysis

### Summary Statistics (numerical columns)

In [ ]:
print("=== PRODUCTS ===")
print(products.describe())


In [ ]:
print("=== SALES ===")
print(sales.describe())


In [ ]:
print("=== INVENTORY ===")
print(inventory.describe())


### Missing Value Check

In [ ]:
def missing_report(df, name):
    miss = df.isnull().sum()
    pct  = (miss / len(df) * 100).round(1)
    tbl  = pd.DataFrame({'missing_count': miss, 'missing_%': pct})
    tbl  = tbl[tbl['missing_count'] > 0]
    print(f"\n── {name}  ({df.shape[0]} rows) ─────────────────────────")
    if tbl.empty:
        print("  No missing values.")
    else:
        print(tbl.to_string())

for df, name in [(customers,'CUSTOMERS'), (products,'PRODUCTS'), (stores,'STORES'),
                 (sales,'SALES'), (inventory,'INVENTORY')]:
    missing_report(df, name)


### Data Types

In [ ]:
for df, name in [(customers,'customers'), (products,'products'), (stores,'stores'),
                 (sales,'sales'), (inventory,'inventory')]:
    print(f"\n── {name}.dtypes ───────────────────────")
    print(df.dtypes.to_string())


## 1.3 – Basic Information Retrieval

### How many unique products are in the product catalog?

In [ ]:
unique_products = products['product_id'].nunique()
print(f"Unique products in the catalog: {unique_products}")


### Top 5 most expensive products

In [ ]:
top5 = (products[['product_name', 'category', 'brand', 'price']]
        .nlargest(5, 'price')
        .reset_index(drop=True))
print("Top 5 Most Expensive Products:")
print(top5.to_string(index=False))


### Which store has the largest floor space?

In [ ]:
largest = stores.loc[stores['size_sqft'].idxmax()]
print(f"Store with Largest Floor Space")
print(f"  Name      : {largest['store_name']}")
print(f"  City/State: {largest['city']}, {largest['state']}")
print(f"  Size      : {largest['size_sqft']:,.0f} sq ft")


### Distribution of customers by state

In [ ]:
state_dist = (customers['state']
              .value_counts()
              .rename_axis('state')
              .reset_index(name='count'))
print("Customer Distribution by State (raw):")
print(state_dist.to_string(index=False))


---
# Part 2: Data Cleaning

## 2.1 – Identifying All Missing Values

In [ ]:
def full_missing_report(df, name):
    total_cells   = df.shape[0] * df.shape[1]
    total_missing = df.isnull().sum().sum()
    pct           = total_missing / total_cells * 100
    print(f"\n{'='*55}")
    print(f"  {name}  |  {total_missing} missing / {total_cells} cells ({pct:.1f}%)")
    print(f"{'='*55}")
    for col in df.columns:
        n = df[col].isnull().sum()
        if n > 0:
            print(f"  {col:<22} {n:>3} missing  ({df[col].dtype})")

for df, name in [(customers,'CUSTOMERS'), (products,'PRODUCTS'), (stores,'STORES'),
                 (sales,'SALES'), (inventory,'INVENTORY')]:
    full_missing_report(df, name)


## 2.2 – Imputing Numerical Missing Values (column mean)

In [ ]:
def impute_numeric(df, name):
    num_cols = df.select_dtypes(include='number').columns
    for col in num_cols:
        n_miss = df[col].isnull().sum()
        if n_miss > 0:
            mean_val = df[col].mean()
            df[col]  = df[col].fillna(mean_val)
            print(f"  {name}.{col:<22} {n_miss} cell(s) → mean = {mean_val:.2f}")
    return df

print("Numerical imputations:")
customers = impute_numeric(customers, 'customers')
products  = impute_numeric(products,  'products')
stores    = impute_numeric(stores,    'stores')
sales     = impute_numeric(sales,     'sales')
inventory = impute_numeric(inventory, 'inventory')


## 2.3 – Imputing Categorical Missing Values (mode)

In [ ]:
def impute_categorical(df, name):
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        n_miss = df[col].isnull().sum()
        if n_miss > 0:
            mode_val = df[col].mode()[0]
            df[col]  = df[col].fillna(mode_val)
            print(f"  {name}.{col:<22} {n_miss} cell(s) → mode = '{mode_val}'")
    return df

print("Categorical imputations:")
customers = impute_categorical(customers, 'customers')
products  = impute_categorical(products,  'products')
stores    = impute_categorical(stores,    'stores')
sales     = impute_categorical(sales,     'sales')
inventory = impute_categorical(inventory, 'inventory')


## 2.4 – Imputing Date Columns (forward fill / backward fill)

In [ ]:
def impute_dates(df, name, date_cols):
    """
    Strategy: forward fill first (use the chronologically prior date),
    then backward fill as a fallback for any leading rows with no prior date.
    """
    for col in date_cols:
        n_miss = df[col].isnull().sum()
        if n_miss > 0:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = df[col].ffill().bfill()
            df[col] = df[col].dt.strftime('%Y-%m-%d')
            print(f"  {name}.{col}: {n_miss} date(s) filled via ffill/bfill")
    return df

print("Date imputations:")
customers = impute_dates(customers, 'customers', ['registration_date'])
stores    = impute_dates(stores,    'stores',    ['opening_date'])
sales     = impute_dates(sales,     'sales',     ['date'])
inventory = impute_dates(inventory, 'inventory', ['last_restock_date'])


## 2.5 – Removing Duplicates

### Approach for Identifying Duplicates

A duplicate is a row where **every field** is identical, indicating the same record
was accidentally entered more than once.

- **`customers`** – checked on all columns; a customer sharing all attributes with
  another row is an unambiguous data entry repeat.
- **`products`** – checked on all columns; identical `product_id`, name, price, etc.
  across two rows confirms a duplicate entry.

We use `keep='first'` to retain the first occurrence and drop subsequent ones.


In [ ]:
for df, name in [(customers, 'CUSTOMERS'), (products, 'PRODUCTS')]:
    before = len(df)
    df_clean = df.drop_duplicates(keep='first').reset_index(drop=True)
    removed  = before - len(df_clean)
    print(f"{name}:  {before} rows → {len(df_clean)} rows  ({removed} duplicate(s) removed)")
    if name == 'CUSTOMERS':
        customers = df_clean
    else:
        products  = df_clean


---
## Discussion Questions

### 1. Key advantages of Pandas for data cleaning vs. other methods

Pandas offers several major advantages over alternatives like Excel, SQL, or raw Python:

- Expressive, readable API: Operations like `fillna()`, `drop_duplicates()`, and
  `str.replace()` are concise one-liners. What would take dozens of Excel clicks is a
  single line of code.
- Vectorised performance: Operations run on entire columns at once (backed by NumPy),
  making them orders of magnitude faster than row-by-row loops.
- Reproducibility: Every cleaning step is captured in code. Unlike a GUI tool, a
  notebook can be re-run identically on updated data at any time.
- Integrated exploration: `describe()`, `isnull()`, and `value_counts()` let you
  discover quality issues and fix them in the same environment.
- Ecosystem integration: Cleaned DataFrames feed directly into Matplotlib/Seaborn for
  visualisation, Scikit-learn for modelling, or SQL databases for storage—no file
  conversion needed.

---

### 2. Handling missing values when the pattern is non-random (MNAR / MAR)

When missingness has a pattern, the reason *why* data is missing carries information:

| Scenario | Example from this dataset | Better strategy |
|----------|--------------------------|-----------------|
| MNAR | Customers who don't want marketing skip email | Add `email_provided` flag; don't impute with a fake email |
| MAR | Older stores missing `size_sqft` because records weren't digital | Impute within the same region's mean, not the global mean |
| Structural | `reorder_level = NaN` for a discontinued product | Do not impute; treat as intentional absence |
| Temporal gap | `last_restock_date` missing because store just opened | Backward fill from first available date, not forward fill |

Blindly applying global column-mean imputation in these cases would **introduce bias**.
Always investigate *why* data is missing before choosing a strategy.

---

### 3. Data quality issues not visible through simple inspection

This dataset is a perfect illustration — `isnull()` reported only **32 missing cells**,
yet the data had many additional problems:

| Issue | Example found in this data |
|-------|---------------------------|
| Case inconsistency | `'Electronics'` vs `'electronics'` vs `'ELECTRONICS'` — all non-null |
| Synonym variants | `'Home & Kitchen'` vs `'Home and Kitchen'` — different strings, same meaning |
| Abbreviated vs. full values | `'CA'` vs `'California'` in the state column |
| Abbreviation sprawl | `'CC'`, `'Visa'`, `'Credit'`, `'CREDIT CARD'` all meaning Credit Card |
| Inconsistent phone formats | `555-123-4567`, `212.555.6789`, `(555) 987-6543` |
| Zip code format variance | `'90001'` vs `'90001-1234'` (with ZIP+4 suffix) |
| Cross-field inconsistency | `unit_price × quantity ≠ total` for some sales rows |

---

### 4. Documenting the data cleaning process for reproducibility

Best practices used in this notebook and beyond:

1. Version control with Git: Commit after each logical phase with descriptive messages
   such as `"Part 2.3 – impute categorical columns with mode"`.
2. Capture provenance: Record the source filename, row count, and SHA-256 hash of the
   raw CSV at the top before any transformation.
3. Explanatory Markdown above every code block: Explain *why* a decision was made
   (e.g., "ffill for dates because records are time-ordered"), not just *what* was done.
4. Audit-trail helper functions: The `impute_numeric`, `impute_categorical`, and
   `impute_dates` functions in this notebook print exactly what was changed and why—this
   log is part of the documentation.
5. Saved intermediate checkpoints: Export a `_cleaned.csv` after each major phase so
   issues can be traced to a specific step.
6. Data dictionary: Maintain a separate README or spreadsheet describing each column,
   its expected type, valid value range, and known caveats.

---

### 5. When to remove rows with missing values rather than imputing

Imputation is not always the right answer. Prefer **dropping rows** when:

- Missingness rate is very high (e.g., >40–50% of a column): imputing would
  essentially invent most of that column's data.
- The column is the target / outcome variable: Imputing `total` in a revenue model
  would leak fabricated signal into what we're trying to predict.
- The affected rows are a tiny fraction of a large dataset (<1–2%) and removing them
  won't meaningfully reduce statistical power.
- Business rules make the row invalid: In this dataset, a sale with no `store_id`
  "and" no `date` cannot be placed in any analysis—imputing both fields would create a
  fictional transaction.
- Time-series integrity: A transaction with a missing date cannot be reliably assigned
  to a period; forward fill may place it in the wrong reporting month.
